# I have made it such, that you can use the LLM responses provided to recreate the SFT files for fine-tuning that I used for experiments.

## Thus, you do not have to use your own money to generate responses from teacher models.

## This allows you to finetune your own bundle generation model.

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np
import csv
import pickle
import pickle as pkl
import copy
import re
import random
import matplotlib.pyplot as plt
import itertools
import json
import openai
import time
import sys
import pickle as pkl
import os

!pip install ipython-autotime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.1 MB/s eta 0:00:00


In [19]:
!rm -rf LLM4BEAR
!git clone --depth 1 --filter=blob:none --sparse https://github.com/anon5159753/LLM4BEAR.git
!cd LLM4BEAR && git sparse-checkout set "4_Bundle Generation"

Cloning into 'LLM4BEAR'...
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 49 (delta 1), reused 15 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (49/49), 27.10 KiB | 1.50 MiB/s, done.
Resolving deltas: 100% (1/1), done.
remote: Enumerating objects: 1, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1 (delta 0), reused 1 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (1/1), 67 bytes | 67.00 KiB/s, done.
remote: Enumerating objects: 190, done.
remote: Counting objects: 100% (190/190), done.
remote: Compressing objects: 100% (188/188), done.
remote: Total 190 (delta 46), reused 2 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (190/190), 29.39 MiB | 5.37 MiB/s, done.
Resolving deltas: 100% (46/46), done.
Updating files: 100% (194/194), done.


In [5]:

import os

path = "/content/drive/MyDrive/baselines/SFT/"

try:
    os.makedirs(path, exist_ok=True)
    print(f"Successfully created: {path}")
except Exception as e:
    print(f"An error occurred: {e}")



Successfully created: /content/drive/MyDrive/baselines/SFT/


In [ ]:
import asyncio
from openai import AsyncOpenAI

from google.colab import userdata


open_secret_key = userdata.get('open_router')

if open_secret_key:
  print("OpenRouter Token retrieved successfully.")
else:
  print("Token not found in Colab Secrets.")


async_client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=open_secret_key,
)



In [6]:
def get_zero_shot(data_info):

    split_test = data_info.split('|split|')
    empty = ""
    for i in range(len(split_test)):
        empty += "product" + str(i + 1) + ". " + split_test[i] + "\n"

    test_prompts = """A bundle can be a set of alternative or complementary products that are purchased with a certain intent.\nPlease detect bundles from a sequence of products. Each bundle must contain multiple products.\nDetect bundles for the below product sequence:\n\n""" + empty + "\n"





    output_format = """Try to infer any intents that may exist in the list of products if purchased together, each intent should help form a bundle with associated products.
Each bundle should have more than a single product. Use product numbers rather than item names. There could be a single or multiple bundles present.

**## OUTPUT FORMAT:**
After your intent analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that contains your bundles.

**JSON Schema:**
```json
{{
  "bundle1": ["product number", "product number", "product number", "..."],
  "bundle2": ["product number", "product number"],
  "bundleN": ["product number", "product number", "..."]
}}
````
Think step-by-step."
"""
    test_prompts += output_format
    return test_prompts

def get_zero_shot_prompts(data_info):

    adding_metrics = "\nFunctionality Integration: Describe how a user would utilize this collection of items to achieve their primary goal. Considering the entire workflow, is this a complete and logical set of items for the task, or is there an irrelevant or missing item? \n"\
              "Similarity: What is the common theme or category that connects these items?\n"\
              "Complementarity: Are these items more valuable together than they would be if sold separately? Does the presence of one item create a clear reason to buy the other(s)?\n"\
              "Diversity: Does the variety of items in this bundle cater to a broad set of related needs for a single user, or does the mix of items seem unfocused and random?\n"

    split_test = data_info.split('|split|')
    empty = ""
    for i in range(len(split_test)):
        empty += "product" + str(i + 1) + ". " + split_test[i] + "\n"

    test_prompts = """A bundle can be a set of alternative or complementary products that are purchased with a certain intent.\nPlease detect bundles from a sequence of products. Each bundle must contain multiple products.\nDetect bundles for the below product sequence:\n\n""" + empty + "\n"





    output_format = [f"""Try to infer any intents that may exist in the list of products if purchased together, each intent should help form a bundle with associated products. Each bundle should have more than a single product. Provide **CONCISE** analysis of the following metrics.
    {adding_metrics}
There could be a single or multiple bundles present and some bundles can be subsets of the others.

**## OUTPUT FORMAT:**
After your analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that contains your bundles. Use product numbers rather than item names. Bundle intents must be 3 words or less.

**JSON Schema:**
```json
{{
  "intent_label_1": ["1", "3"],
  "intent_label_2": ["2", "4", "5", "..."]
}}
````
In your analysis, refer to products using extremely concise descriptors (e.g., 'tablet case' instead of 'Supcase New Tablet Case Protection').
Think step-by-step.""",
"""Each bundle should have more than a single product. There could be a single or multiple bundles present and some bundles can be subsets of the others. Provide **CONCISE** analysis:
1. Try to infer any intents that may exist in the list of products if purchased together, each intent should help form a bundle with associated products.
2. Using the inferred intents, group items by thematic similarity to the intent.
3. Think about how each item complements and adds value as a part of an intent/thematic group in order to fulfill a single primary goal and form a bundle. Exclude less relevant items.
4. See if the adding an additional item outside the established bundle increases the perceived value of the bundle.

**## OUTPUT FORMAT:**
After your analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that contains your bundles. Use product numbers rather than item names. Bundle intents must be 3 words or less.

**JSON Schema:**
```json
{
  "intent_label_1": ["1", "3"],
  "intent_label_2": ["2", "4", "5", "..."]
}
````
In your analysis, refer to products using extremely concise descriptors (e.g., 'tablet case' instead of 'Supcase New Tablet Case Protection')
Think step-by-step.""",
"""
**STEP-BY-STEP REASONING:**
Before generating the JSON for bundles, perform a deep-dive but **CONCISE** analysis on the following metrics:
1. **Functionality Integration & Goal Completeness:** Describe how a user utilizes this collection to achieve a primary goal. Is this a complete, logical set for the task, or is there an irrelevant "fluff" item or a "missing link"?
2. **Complementarity & Synergy:** Explain the value gap. Does owning Item A make Item B significantly more useful? Ensure these items are more valuable together than separately.
3. **Similarity & Thematic Cohesion:** What is the common theme or category? Identify the single, clear "User Persona" (e.g., 'The Weekend Hiker') that connects these items.
4. **Diversity & Sub-Bundle Logic:** Ensure the variety caters to a broad set of related needs for one user without being unfocused. Identify if small 2-item bundles are actually subsets of a larger primary bundle.

**CONSTRAINTS:**
- Each bundle MUST contain at least 2 products.
- Do NOT include items that are only vaguely related (e.g., don't put a 'Yoga Mat' in a 'Running Kit' just because both are 'Fitness').

**## OUTPUT FORMAT:**
After your analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that contains your bundles. Use product numbers rather than item names. Bundle intents must be 3 words or less.

**JSON Schema:**
{
  "intent_label_1": ["1", "3"],
  "intent_label_2": ["2", "4", "5", "..."]
}
In your analysis, refer to products using extremely concise descriptors (e.g., 'tablet case' instead of 'Supcase New Tablet Case Protection')
Think step-by-step."""]

    prompts = [test_prompts for _ in range(len(output_format))]

    for i in range(len(prompts)):
        prompts[i] += output_format[i]


    return prompts

In [27]:
async def difficult_run(path, m_name, model, diffs=[]):

    if diffs == []:
        return

    else:
        print("accessing difficult keys")

        dataset = path.split("/")[5]

        domain = path.split("/")[6]

        hard_train = diffs[0]
        hard_val = diffs[1]

        training_set_path = "training_set.npy"
        validation_set_path = "validation_set.npy"

        train_set = np.load(path + training_set_path, allow_pickle=True).tolist()
        val_set = np.load(path + validation_set_path, allow_pickle=True).tolist()


        train_zero_shot_prompts = []
        val_zero_shot_prompts = []

        for i in hard_train:
            train_prompt_1, train_prompt_2, train_prompt_3 = get_zero_shot_prompts(train_set[i])
            train_zero_shot_prompts.append(train_prompt_1)
            train_zero_shot_prompts.append(train_prompt_2)
            train_zero_shot_prompts.append(train_prompt_3)

        for i in hard_val:
            val_prompt_1, val_prompt_2, val_prompt_3 = get_zero_shot_prompts(val_set[i])
            val_zero_shot_prompts.append(val_prompt_1)
            val_zero_shot_prompts.append(val_prompt_2)
            val_zero_shot_prompts.append(val_prompt_3)

        print()
        print("="*80)
        print()
        print(train_zero_shot_prompts[0])
        print()
        print("="*80)
        print()
        print(val_zero_shot_prompts[0])
        print()
        print("="*80)
        print()

        train_zero_shot_prompt_list = [{"prompts": i} for i in train_zero_shot_prompts]
        val_zero_shot_prompt_list = [{"prompts": i} for i in val_zero_shot_prompts]


        system_message = "You are an **Expert E-commerce Analyst and Product Bundler**. Your task is to receive a sequence of products and accurately organise them into logical, desirable bundles."

        train_responses = await openai_request(train_zero_shot_prompt_list, model=model, system=system_message)
        val_responses = await openai_request(val_zero_shot_prompt_list, model=model, system=system_message)

        print("\n",train_responses[0],"\n")

        with open(f"/content/drive/MyDrive/baselines/SFT/{dataset}/{domain}/" + f'{m_name}_train_responses.pkl', 'wb') as f:
            pkl.dump([hard_train, train_responses], f)

        with open(f"/content/drive/MyDrive/baselines/SFT/{dataset}/{domain}/" + f'{m_name}_val_responses.pkl', 'wb') as f:
            pkl.dump([hard_val, val_responses], f)




async def openrouter_run(path):

    dataset = path.split("/")[5]

    domain = path.split("/")[6]

    tasks = []

    training_set_path = "training_set.npy"
    validation_set_path = "validation_set.npy"

    train_set = np.load(path + training_set_path, allow_pickle=True).tolist()
    val_set = np.load(path + validation_set_path, allow_pickle=True).tolist()

    train_keys = list(train_set.keys())
    val_keys = list(val_set.keys())

    train_zero_shot_prompts = []
    val_zero_shot_prompts = []

    for i in train_keys:
        train_prompt_1, train_prompt_2, train_prompt_3 = get_zero_shot_prompts(train_set[i])
        train_zero_shot_prompts.append(train_prompt_1)
        train_zero_shot_prompts.append(train_prompt_2)
        train_zero_shot_prompts.append(train_prompt_3)

    for i in val_keys:
        val_prompt_1, val_prompt_2, val_prompt_3 = get_zero_shot_prompts(val_set[i])
        val_zero_shot_prompts.append(val_prompt_1)
        val_zero_shot_prompts.append(val_prompt_2)
        val_zero_shot_prompts.append(val_prompt_3)

    system_message = "You are an **Expert E-commerce Analyst and Product Bundler**. Your task is to receive a sequence of products and accurately organise them into logical, desirable bundles."

    baseline_models = [
        "google/gemini-2.0-flash-001",
        "anthropic/claude-3-5-haiku",
        "meta-llama/llama-3.3-70b-instruct",
        "mistralai/mistral-small-24b-instruct-2501"
    ]
    names = ["gemini", "claude", "llama", "mistral"]

    batch_sizes = [40, 40, 10, 15]

    train_zero_shot_prompt_list = [{"prompts": i} for i in train_zero_shot_prompts]
    val_zero_shot_prompt_list = [{"prompts": i} for i in val_zero_shot_prompts]

    # start of edit

    async def run_model(model_id, name, batch_size, pos):
        train_responses = await run_experiment(
            train_zero_shot_prompt_list,
            model_id=model_id,
            system=system_message,
            batch_size=batch_size,
            pos=pos
        )

        val_responses = await run_experiment(
            val_zero_shot_prompt_list,
            model_id=model_id,
            system=system_message,
            batch_size=batch_size,
            pos=pos
        )

        with open(f"/content/drive/MyDrive/baselines/SFT/{dataset}/{domain}/" + f'{name}_train_responses.pkl', 'wb') as f:
            pkl.dump(train_responses, f)

        with open(f"/content/drive/MyDrive/baselines/SFT/{dataset}/{domain}/" + f'{name}_val_responses.pkl', 'wb') as f:
            pkl.dump(val_responses, f)



    for i, (model_id, name, batch_size) in enumerate(zip(baseline_models, names, batch_sizes)):
        tasks.append(
            asyncio.create_task(
                run_model(model_id, name, batch_size, i)
            )
        )

    await asyncio.gather(*tasks)

In [ ]:
from tqdm.asyncio import tqdm_asyncio
from tqdm.auto import tqdm

async def openrouter_request(user, model_id, system=None):
    if system:
        message = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    else:
        message = [{"role": "user", "content": user}]

    # Robust retry loop with exponential backoff
    for delay_secs in (2**x for x in range(0, 3)):
        try:
            response = await async_client.chat.completions.create(
                model=model_id, # Now dynamic!
                messages=message,
                temperature=0,
                max_tokens=1200, # Adjust based on bundle length
                # Optional: extra_body is where OpenRouter specific features go
                extra_body={
                    "provider": {"require_parameters": True}
                }
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            randomness_collision_avoidance = random.randint(0, 1000) / 1000.0
            sleep_dur = delay_secs + randomness_collision_avoidance
            print(f"Error with {model_id}: {e}. Retrying in {round(sleep_dur, 2)}s.")
            await asyncio.sleep(sleep_dur)

    return None



async def run_experiment(prompts, model_id, system=None, batch_size=20, inter_batch_delay=5, pos=0):
    """
    Processes requests in explicit batches to avoid 'Request Timed Out' errors.
    """
    results = []
    total_prompts = len(prompts)
    print(f"🚀 Initializing experiment for: {model_id}")

    # Use a standard tqdm bar for the batches
    pbar = tqdm(total=total_prompts, desc=f"📊 {model_id.split('/')[-1]}", position=pos, leave=True)

    for i in range(0, total_prompts, batch_size):
        batch = prompts[i : i + batch_size]

        # 1. Create tasks for just THIS batch
        tasks = [
            openrouter_request(d["prompts"], model_id, system=system)
            for d in batch
        ]

        # 2. Gather ONLY this batch and wait for it to finish
        batch_results = await asyncio.gather(*tasks)
        results.extend(batch_results)

        # 3. Update the progress bar
        pbar.update(len(batch))

        # 4. Mandatory cooldown to prevent saturation
        if i + batch_size < total_prompts:
            # print(f"  [Batch Done] Sleeping {inter_batch_delay}s to avoid timeouts...")
            await asyncio.sleep(inter_batch_delay)

    pbar.close()
    # print(f"✅ {model_id} — All {len(results)} requests completed.\n")
    return results

# Responses have already been provided, so you can skip this step.

In [ ]:
electronic_bundlerec_path = '/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/electronic/'
clothing_bundlerec_path = '/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/clothing/'
food_bundlerec_path = '/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/food/'

electronic_llm4bear_path = '/content/LLM4BEAR/4_Bundle Generation/data/llm4bear/electronic/'
clothing_llm4bear_path = '/content/LLM4BEAR/4_Bundle Generation/data/llm4bear/clothing/'
food_llm4bear_path = '/content/LLM4BEAR/4_Bundle Generation/data/llm4bear/food/'



# ==========================================
# EXECUTE ORDER 66
# ==========================================

# print("🚀 STARTING FULL BATCH EXECUTION (6 DATASETS)...\n")

# # --- 1. ELECTRONIC (BundleRec) ---

# print("--------------------------------------------------\n")
# print("💻 [1/6] Running ELECTRONIC BundleRec...\n")
# await openrouter_run(electronic_bundlerec_path)
# print("✅ Electronic BundleRec Finished.")




# # --- 2. CLOTHING (BundleRec) ---

# print("\n--------------------------------------------------\n")
# print("👕 [2/6] Running CLOTHING BundleRec...\n")
# await openrouter_run(clothing_bundlerec_path)
# print("✅ Clothing BundleRec Finished.")


# # # --- 3. FOOD (BundleRec) ---

# print("\n--------------------------------------------------\n")
# print("🍔 [3/6] Running FOOD BundleRec...\n")
# await openrouter_run(food_bundlerec_path)
# print("✅ Food BundleRec Finished.")


# # --- 4. ELECTRONIC (LLM4BEAR) ---

# print("\n--------------------------------------------------\n")
# print("💻 [4/6] Running ELECTRONIC LLM4BEAR...\n")
# await openrouter_run(electronic_llm4bear_path)
# print("✅ Electronic LLM4BEAR Finished.")


# # --- 5. CLOTHING (LLM4BEAR) ---

# print("\n--------------------------------------------------\n")
# print("👕 [5/6] Running CLOTHING LLM4BEAR...\n")
# await openrouter_run(clothing_llm4bear_path)
# print("✅ Clothing LLM4BEAR Finished.")

# # --- 6. FOOD (LLM4BEAR) ---

# print("\n--------------------------------------------------\n")
# print("🍔 [6/6] Running FOOD LLM4BEAR...\n")
# await openrouter_run(food_llm4bear_path)
# print("✅ Food LLM4BEAR Finished.")


# print("\n🎉🎉🎉 ALL 6 EXPERIMENTS COMPLETED 🎉🎉🎉")


# If you can run and get the SFT file for fine-tuning to your own Google Drive, otherwise, you can do SFT with the files provided already.

In [7]:

def extract_json_simple_replace(response_text):
    if response_text is None:
        return None

    try:
        # 1. Use a case-insensitive split or check for the separator
        if "===JSON_START===" not in response_text:
            # Fallback: Try to find the first '{' anyway
            json_part = response_text
        else:
            json_part = response_text.split("===JSON_START===")[1]

        # 2. Find the boundaries
        first_brace = json_part.find('{')
        last_brace = json_part.rfind('}')

        if first_brace == -1 or last_brace == -1:
            return None

        # 3. Extract and clean
        json_string = json_part[first_brace : last_brace + 1].strip()

        # 4. Parse
        return json.loads(json_string)

    except Exception as e:
        # print(f"Extraction error: {e}")
        return None

def convert_products_to_indices(product_names):
    """
    Converts product identifiers (e.g., 'product2', '2', or 2)
    into zero-based integer indices (N-1).
    """
    indices = []
    # Regex to find the digits anywhere in the string
    pattern = re.compile(r'(\d+)')

    for name in product_names:
        # Convert to string to handle raw integers like 2 or 4
        name_str = str(name).strip()

        match = pattern.search(name_str)
        if match:
            # Extract the number
            product_number = int(match.group(1))
            # Convert to zero-based index
            index = product_number - 1
            indices.append(index)

    return indices


def compute_jaccard_reward(prediction_str, ground_truth_bundles):
    """
    prediction_str: Raw LLM output string
    ground_truth_bundles: List of lists of indices, e.g. [[0, 1, 4], [3, 5]]
    """
    try:
        anti_reward = 0
        # 1. Extract JSON from the ===JSON_START=== separator
        prediction_dict = extract_json_simple_replace(prediction_str)

        # 2. Convert all predicted bundles into sets of indices
        pred_sets = []
        for bundle_key, product_nums in prediction_dict.items():
            # e.g., converts ["product1", "product2"] to {0, 1}
            indices = convert_products_to_indices(product_nums)
            if indices and len(indices) >= 2:
                pred_sets.append(set(indices))
            if len(indices) == 1:
                anti_reward += -0.15

        if not pred_sets:
            return -0.4

        # Convert ground truth lists to sets
        gt_sets = [set(gt) for gt in ground_truth_bundles]

        # 3. Greedy Matching: Assign the best prediction to each GT
        all_matches = []
        for g_idx, g_set in enumerate(gt_sets):
            for p_idx, p_set in enumerate(pred_sets):
                intersection = len(g_set & p_set)
                union = len(g_set | p_set)
                score = intersection / union if union > 0 else 0
                if score > 0:
                    all_matches.append((score, g_idx, p_idx))

        # Sort by best Jaccard score first
        all_matches.sort(key=lambda x: x[0], reverse=True)

        assigned_gts = set()
        assigned_ps = set()
        reward_sum = 0

        for score, g_idx, p_idx in all_matches:
            if g_idx not in assigned_gts and p_idx not in assigned_ps:
                assigned_gts.add(g_idx)
                assigned_ps.add(p_idx)
                reward_sum += score

        # Final Reward: Average Jaccard across all Ground Truths
        # (This forces the model to cover ALL ground truth bundles, not just one)
        return reward_sum / max(len(gt_sets), 1*len(pred_sets)) + anti_reward # was 0.65 before

    except Exception as e:
        # Penalize formatting errors to teach the model JSON structure
        return -0.5

In [23]:
def compute_jaccards_for_all(path):

    domain = path.split("/")[6]
    dataset = path.split("/")[5]

    train_ground_indices_path = "train_ground_indices.pkl"
    val_ground_indices_path = "val_ground_indices.pkl"

    with open(path + train_ground_indices_path, 'rb') as f:
        ground_indices_train = pkl.load(f)

    with open(path + val_ground_indices_path, 'rb') as f:
        ground_indices_val = pkl.load(f)

    train_keys = list(ground_indices_train.keys())
    val_keys = list(ground_indices_val.keys())

    train_len = len(ground_indices_train)
    val_len = len(ground_indices_val)

    train_guys = [[] for _ in range(train_len)]
    val_guys = [[] for _ in range(val_len)]

    train_jaccard = [[] for _ in range(train_len)]
    val_jaccard = [[] for _ in range(val_len)]

    names = ["4o-mini", "gemini", "claude", "llama", "mistral"]

    for i in range(5):

        # with open(f"/content/drive/MyDrive/baselines/SFT/{dataset}/{domain}/" + f'{names[i]}_train_responses.pkl', 'rb') as f:
        with open(path + "responses/" + f'{names[i]}_train_responses.pkl', 'rb') as f:
            train_responses = pkl.load(f)

        # with open(f"/content/drive/MyDrive/baselines/SFT/{dataset}/{domain}/" + f'{names[i]}_val_responses.pkl', 'rb') as f:
        with open(path + "responses/" + f'{names[i]}_val_responses.pkl', 'rb') as f:
            val_responses = pkl.load(f)

        for j in range(train_len):

            for k in range(3):

                train_response = train_responses[j*3 + k]

                # print(train_response)
                # print(j)
                # print(ground_indices_train[j])

                train_guys[j].append(train_response)

                train_jaccard[j].append(compute_jaccard_reward(train_response, ground_indices_train[train_keys[j]]))

        for j in range(val_len):

            for k in range(3):

                val_response = val_responses[j*3 + k]

                val_guys[j].append(val_response)
                val_jaccard[j].append(compute_jaccard_reward(val_response, ground_indices_val[val_keys[j]]))


    for i in range(train_len):
        # sort order of responses by jaccard sort
        zipped = zip(train_jaccard[i], train_guys[i])
        sorted_pairs = sorted(zipped, key=lambda x: x[0], reverse=True)
        train_jaccard[i], train_guys[i] = map(list, zip(*sorted_pairs))

    for i in range(val_len):
        # sort order of responses by jaccard sort
        zipped = zip(val_jaccard[i], val_guys[i])
        sorted_pairs = sorted(zipped, key=lambda x: x[0], reverse=True)
        val_jaccard[i], val_guys[i] = map(list, zip(*sorted_pairs))

    save_dir = f"/content/drive/MyDrive/baselines/SFT/{dataset}/{domain}/"
    os.makedirs(save_dir, exist_ok=True)


    with open(f"/content/drive/MyDrive/baselines/SFT/{dataset}/{domain}/" + 'train_jaccard.pkl', 'wb') as f:
        pkl.dump([train_guys, train_jaccard], f)

    with open(f"/content/drive/MyDrive/baselines/SFT/{dataset}/{domain}/" + 'val_jaccard.pkl', 'wb') as f:
        pkl.dump([val_guys, val_jaccard], f)


import random


def generate_SFT_samples(path, high_thresh=0.8, low_thresh=0.3, num_pairs=2):

    domain = path.split("/")[6]
    dataset = path.split("/")[5]

    train_ground_indices_path = "train_ground_indices.pkl"
    val_ground_indices_path = "val_ground_indices.pkl"

    with open(path + train_ground_indices_path, 'rb') as f:
        ground_indices_train = pkl.load(f)

    with open(path + val_ground_indices_path, 'rb') as f:
        ground_indices_val = pkl.load(f)

    training_set_path = "training_set.npy"
    validation_set_path = "validation_set.npy"

    train_set = np.load(path + training_set_path, allow_pickle=True).tolist()
    val_set = np.load(path + validation_set_path, allow_pickle=True).tolist()

    train_keys = list(train_set.keys())
    val_keys = list(val_set.keys())

    train_len = len(train_keys)
    val_len = len(val_keys)

    train_jac_path = "train_jaccard.pkl"
    val_jac_path = "val_jaccard.pkl"

    with open(f"/content/drive/MyDrive/baselines/SFT/{dataset}/{domain}/" + train_jac_path, 'rb') as f:
        train_guys, train_jaccards = pkl.load(f)

    with open(f"/content/drive/MyDrive/baselines/SFT/{dataset}/{domain}/" + val_jac_path, 'rb') as f:
        val_guys, val_jaccards = pkl.load(f)


    train_higher_than_threshold = 0
    val_higher_than_threshold = 0
    train_unique = 0
    val_unique = 0

    SFT_samples = []
    val_SFT_samples = []

    train_diff = []
    val_diff = []

    for i in range(train_len):
        # Identify all potential 'Chosen' candidates
        chosen_indices = [idx for idx, s in enumerate(train_jaccards[i]) if s >= high_thresh]

        if len(chosen_indices) > 0:
            train_higher_than_threshold += 1
        else:
            train_diff.append(train_keys[i])

        # Identify all potential 'Rejected' candidates (excluding -0.5 syntax errors)
        rejected_indices = [idx for idx, s in enumerate(train_jaccards[i]) if s < low_thresh and s != -0.5]



        if len(rejected_indices) == 0 and train_jaccards[i][0] > 0.8:
            if random.random() < 0.5:
                SFT_samples.append({
                        "prompt": get_zero_shot(train_set[train_keys[i]]),
                        "answer": train_guys[i][0],
                        "score": train_jaccards[i][0] # Useful for analysis
                    })


        reversed_rejected = rejected_indices[::-1]

        # Generate the Cartesian product of all valid pairs
        pairs_added = 0
        for c_idx in chosen_indices:
            if pairs_added >= num_pairs: break # Keep it balanced!
            SFT_samples.append({
                "prompt": get_zero_shot(train_set[train_keys[i]]),
                "answer": train_guys[i][c_idx],
                "score": train_jaccards[i][c_idx] # Useful for analysis
            })
            if pairs_added == 0:
                train_unique += 1

            pairs_added += 1


    for i in range(val_len):
        # Identify all potential 'Chosen' candidates
        chosen_indices = [idx for idx, s in enumerate(val_jaccards[i]) if s >= high_thresh]

        if len(chosen_indices) > 0:
            val_higher_than_threshold += 1
        else:
            val_diff.append(val_keys[i])

        # Identify all potential 'Rejected' candidates (excluding -0.5 syntax errors)
        rejected_indices = [idx for idx, s in enumerate(val_jaccards[i]) if s < low_thresh and s != -0.5]


        if len(rejected_indices) == 0 and val_jaccards[i][0] > 0.8:
            val_SFT_samples.append({
                    "prompt": get_zero_shot(val_set[val_keys[i]]),
                    "answer": val_guys[i][0],
                    "score": val_jaccards[i][0] # Useful for analysis
                })

        reversed_rejected = rejected_indices[::-1]

        # Generate the Cartesian product of all valid pairs
        pairs_added = 0
        for c_idx in chosen_indices:

            if pairs_added >= num_pairs: break # Keep it balanced!
            val_SFT_samples.append({
                "prompt": get_zero_shot(val_set[val_keys[i]]),
                "answer": val_guys[i][c_idx],
                "score": val_jaccards[i][c_idx] # Useful for analysis
            })
            if pairs_added == 0:
                val_unique += 1
            pairs_added += 1

    try:

        # with open(f"/content/drive/MyDrive/baselines/SFT/{dataset}/{domain}/" f'gpt-5.2_train_responses.pkl', 'rb') as f: # or whatever model you used for difficult keys
        with open(path + "responses/" + f'gpt-5.2_train_responses.pkl', 'rb') as f:
            hard_train_keys, hard_train_responses = pkl.load(f)

        # with open(f"/content/drive/MyDrive/baselines/SFT/{dataset}/{domain}/" f'gpt-5.2_val_responses.pkl', 'rb') as f: # or whatever model you used for difficult keys
        with open(path + "responses/" + f'gpt-5.2_val_responses.pkl', 'rb') as f:
            hard_val_keys, hard_val_responses = pkl.load(f)

    except:

        print("Haven't yet ran hard problems.")

    hard_additions = 0
    hard_unique = 0

    for j in range(len(hard_train_keys)):
        key_added_already = False
        for k in range(3):
            hard_train_response = hard_train_responses[j*3 + k]
            hard_train_score = compute_jaccard_reward(hard_train_response, ground_indices_train[hard_train_keys[j]])
            if hard_train_score > 0.7: # they are hard problems, so to allow harder bundling knowledge to be more represented, reduce score, and add three times.
                for _ in range(3):
                    hard_additions += 1
                    SFT_samples.append({
                    "prompt": get_zero_shot(train_set[hard_train_keys[j]]),
                    "answer": hard_train_response,
                    "score": hard_train_score # Useful for analysis
                })
                if not key_added_already:
                    hard_unique += 1
                    key_added_already = True # Only count this key once

    for j in range(len(hard_val_keys)):
        key_added_already = False # Use a flag for the key
        for k in range(3):
            hard_val_response = hard_val_responses[j*3 + k]
            # print(hard_val_response)
            hard_val_score = compute_jaccard_reward(hard_val_response, ground_indices_val[hard_val_keys[j]])
            if hard_val_score > 0.7: # they are hard problems, so to allow harder bundling knowledge to be more represented, reduce score, and add three times. Also, send it to the training set.
                for _ in range(3):
                    hard_additions += 1
                    SFT_samples.append({
                    "prompt": get_zero_shot(val_set[hard_val_keys[j]]),
                    "answer": hard_val_response,
                    "score": hard_val_score # Useful for analysis
                })
                if not key_added_already:
                    hard_unique += 1
                    key_added_already = True # Only count this key once


    random.shuffle(SFT_samples)
    random.shuffle(val_SFT_samples)


    print(path)
    print(f"Train higher than threshold: {train_higher_than_threshold}/{train_len}, Val higher than threshold: {val_higher_than_threshold}/{val_len}")
    print(f"Train unique: {train_unique}, Val unique: {val_unique}")
    print(f"Hard unique Additions: {hard_unique}")
    print(f"Hard SFT Additions: {hard_additions}")
    print(f"Train SFT samples: {len(SFT_samples)}")
    print(f"Val SFT samples: {len(val_SFT_samples)}")

    return SFT_samples, val_SFT_samples, [train_diff, val_diff]


import json


def save_SFT_jsonl(path, SFT_samples, val_SFT_samples):

    domain = path.split("/")[6]
    dataset = path.split("/")[5]
    system_message = "You are an **Expert E-commerce Analyst and Product Bundler**. Your task is to receive a sequence of products and accurately organise them into logical, desirable bundles."

    # Save SFT Samples (The Expert Teacher)
    with open(f"/content/drive/MyDrive/baselines/SFT/{dataset}/{domain}/" + "comprehensive_train_sft_data.jsonl", 'w') as f:
        for s in SFT_samples:
            entry = {
                "messages": [
                    {"role": "system", "content": system_message},
                    {"role": "user", "content": s["prompt"]},
                    {"role": "assistant", "content": s["answer"]}
                ]
            }
            f.write(json.dumps(entry) + '\n')

    with open(f"/content/drive/MyDrive/baselines/SFT/{dataset}/{domain}/" + "comprehensive_val_sft_data.jsonl", 'w') as f:
        for s in val_SFT_samples:
            entry = {
                "messages": [
                    {"role": "system", "content": system_message},
                    {"role": "user", "content": s["prompt"]},
                    {"role": "assistant", "content": s["answer"]}
                ]
            }
            f.write(json.dumps(entry) + '\n')


    print(f"✅ Saved {len(SFT_samples)} SFT samples.")
    print(f"✅ Saved {len(val_SFT_samples)} Validation samples.")


In [24]:
electronic_bundlerec_path = '/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/electronic/'
clothing_bundlerec_path = '/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/clothing/'
food_bundlerec_path = '/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/food/'

electronic_llm4bear_path = '/content/LLM4BEAR/4_Bundle Generation/data/llm4bear/electronic/'
clothing_llm4bear_path = '/content/LLM4BEAR/4_Bundle Generation/data/llm4bear/clothing/'
food_llm4bear_path = '/content/LLM4BEAR/4_Bundle Generation/data/llm4bear/food/'


six_paths = [electronic_bundlerec_path, clothing_bundlerec_path, food_bundlerec_path, electronic_llm4bear_path, clothing_llm4bear_path, food_llm4bear_path]
amount_of_pairs = [1,1,1,1,1,1]

all_diffs = []

for i, path in enumerate(six_paths):

    compute_jaccards_for_all(path)
    sft_samples, val_sft_samples, difficults = generate_SFT_samples(path, high_thresh=0.8, low_thresh=0.3, num_pairs=amount_of_pairs[i])

    print()
    all_diffs.append(difficults)

/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/electronic/
Train higher than threshold: 320/623, Val higher than threshold: 46/90
Train unique: 320, Val unique: 46
Hard unique Additions: 39
Hard SFT Additions: 156
Train SFT samples: 552
Val SFT samples: 65

/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/clothing/
Train higher than threshold: 471/698, Val higher than threshold: 65/90
Train unique: 471, Val unique: 65
Hard unique Additions: 38
Hard SFT Additions: 153
Train SFT samples: 758
Val SFT samples: 99

/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/food/
Train higher than threshold: 394/644, Val higher than threshold: 56/90
Train unique: 394, Val unique: 56
Hard unique Additions: 44
Hard SFT Additions: 180
Train SFT samples: 679
Val SFT samples: 84

/content/LLM4BEAR/4_Bundle Generation/data/llm4bear/electronic/
Train higher than threshold: 251/623, Val higher than threshold: 39/90
Train unique: 251, Val unique: 39
Hard unique Additions: 58
Hard SFT Additions: 2

In [ ]:
from tqdm.asyncio import tqdm_asyncio
from tqdm.auto import tqdm

MAX_CONCURRENT_REQUESTS = 50
semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

async def single_request(user, model, system=None):
    # The semaphore ensures we only have 50 active requests at any time
    async with semaphore:
        if system:
            message = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        else:
            message = [{"role": "user", "content": user}]

        # Robust retry loop with exponential backoff
        for delay_secs in (2**x for x in range(0, 3)):
            try:
                response = await async_client.chat.completions.create(
                    model=model,
                    messages=message,
                    temperature=0,
                    max_completion_tokens=4000,
                )
                return response.choices[0].message.content.strip()
            except openai.OpenAIError as e:
                # Jitter prevents the "thundering herd" problem on retries
                sleep_dur = delay_secs + random.random()
                await asyncio.sleep(sleep_dur)
        return None

async def openai_request(prompts, model, system=None):
    """
    Tier 4 Optimized Requester with Progress Bar.
    """
    print(f"🚀 Launching {len(prompts)} requests on {model}...")

    # Create the task list
    tasks = [
        single_request(d["prompts"], model=model, system=system)
        for d in prompts
    ]

    # Using tqdm.asyncio.tqdm.gather handles both execution and the bar
    # 'total' is inferred from 'tasks'
    results = await tqdm.gather(*tasks, desc="Processing Hard 303 Samples")

    return results

# Use stronger model to take on difficult sessions to bundle.

In [ ]:
electronic_bundlerec_path = '/content/drive/My Drive/AICL/data/edited_bundlerec/electronic/'
clothing_bundlerec_path = '/content/drive/My Drive/AICL/data/edited_bundlerec/clothing/'
food_bundlerec_path = '/content/drive/My Drive/AICL/data/edited_bundlerec/food/'

electronic_llm4bear_path = '/content/drive/My Drive/AICL/data/llm4bear/electronic/'
clothing_llm4bear_path = '/content/drive/My Drive/AICL/data/llm4bear/clothing/'
food_llm4bear_path = '/content/drive/My Drive/AICL/data/llm4bear/food/'

models = ["gpt-5.2"]

# # ==========================================
# # EXECUTE ORDER 66
# # ==========================================

# for i in models:

#     print("🚀 STARTING FULL BATCH EXECUTION (6 DATASETS)...\n")

#     # --- 1. ELECTRONIC (BundleRec) ---

#     print("--------------------------------------------------\n")
#     print("💻 [1/6] Running ELECTRONIC BundleRec...\n")
#     await difficult_run(electronic_bundlerec_path, m_name=i, model=i, diffs=all_diffs[0])
#     print("✅ Electronic BundleRec Finished.")




#     # --- 2. CLOTHING (BundleRec) ---

#     print("\n--------------------------------------------------\n")
#     print("👕 [2/6] Running CLOTHING BundleRec...\n")
#     await difficult_run(clothing_bundlerec_path, m_name=i, model=i, diffs=all_diffs[1])
#     print("✅ Clothing BundleRec Finished.")


#     # # --- 3. FOOD (BundleRec) ---

#     print("\n--------------------------------------------------\n")
#     print("🍔 [3/6] Running FOOD BundleRec...\n")
#     await difficult_run(food_bundlerec_path, m_name=i, model=i, diffs=all_diffs[2])
#     print("✅ Food BundleRec Finished.")


#     # --- 4. ELECTRONIC (LLM4BEAR) ---

#     print("\n--------------------------------------------------\n")
#     print("💻 [4/6] Running ELECTRONIC LLM4BEAR...\n")
#     await difficult_run(electronic_llm4bear_path, m_name=i, model=i, diffs=all_diffs[3])
#     print("✅ Electronic LLM4BEAR Finished.")


#     # --- 5. CLOTHING (LLM4BEAR) ---

#     print("\n--------------------------------------------------\n")
#     print("👕 [5/6] Running CLOTHING LLM4BEAR...\n")
#     await difficult_run(clothing_llm4bear_path, m_name=i, model=i, diffs=all_diffs[4])
#     print("✅ Clothing LLM4BEAR Finished.")

#     # --- 6. FOOD (LLM4BEAR) ---

#     print("\n--------------------------------------------------\n")
#     print("🍔 [6/6] Running FOOD LLM4BEAR...\n")
#     await difficult_run(food_llm4bear_path, m_name=i, model=i, diffs=all_diffs[5])
#     print("✅ Food LLM4BEAR Finished.")


#     print("\n🎉🎉🎉 ALL 6 EXPERIMENTS COMPLETED 🎉🎉🎉")

#     print('\n\n\n')


In [25]:
electronic_bundlerec_path = '/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/electronic/'
clothing_bundlerec_path = '/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/clothing/'
food_bundlerec_path = '/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/food/'

electronic_llm4bear_path = '/content/LLM4BEAR/4_Bundle Generation/data/llm4bear/electronic/'
clothing_llm4bear_path = '/content/LLM4BEAR/4_Bundle Generation/data/llm4bear/clothing/'
food_llm4bear_path = '/content/LLM4BEAR/4_Bundle Generation/data/llm4bear/food/'


six_paths = [electronic_bundlerec_path, clothing_bundlerec_path, food_bundlerec_path, electronic_llm4bear_path, clothing_llm4bear_path, food_llm4bear_path]
amount_of_pairs = [1,1,1,1,1,1]

all_diffs = []

for i, path in enumerate(six_paths):

    compute_jaccards_for_all(path)
    sft_samples, val_sft_samples, difficults = generate_SFT_samples(path, high_thresh=0.8, low_thresh=0.3, num_pairs=amount_of_pairs[i])
    save_SFT_jsonl(path, sft_samples, val_sft_samples)

    print()
    all_diffs.append(difficults)

/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/electronic/
Train higher than threshold: 320/623, Val higher than threshold: 46/90
Train unique: 320, Val unique: 46
Hard unique Additions: 39
Hard SFT Additions: 156
Train SFT samples: 555
Val SFT samples: 65
✅ Saved 555 SFT samples.
✅ Saved 65 Validation samples.

/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/clothing/
Train higher than threshold: 471/698, Val higher than threshold: 65/90
Train unique: 471, Val unique: 65
Hard unique Additions: 38
Hard SFT Additions: 153
Train SFT samples: 752
Val SFT samples: 99
✅ Saved 752 SFT samples.
✅ Saved 99 Validation samples.

/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/food/
Train higher than threshold: 394/644, Val higher than threshold: 56/90
Train unique: 394, Val unique: 56
Hard unique Additions: 44
Hard SFT Additions: 180
Train SFT samples: 691
Val SFT samples: 84
✅ Saved 691 SFT samples.
✅ Saved 84 Validation samples.

/content/LLM4BEAR/4_Bundle Generation/data/llm4b

In [26]:
def create_SFT_experimental_files(base_path):
    experiments = {
        "BundleRec": ["bundlerec/electronic/", "bundlerec/clothing/", "bundlerec/food/"],
        "LLM4Bear": ["llm4bear/electronic/", "llm4bear/clothing/", "llm4bear/food/"]
    }

    for exp_name, folders in experiments.items():
        train_sft_pool = []
        val_sft_pool = []


        for folder in folders:
            full_path = os.path.join(base_path, folder)

            # Load and merge SFT
            with open(full_path + "comprehensive_train_sft_data.jsonl", 'r') as f:
                train_sft_pool.extend([json.loads(line) for line in f])

            with open(full_path + "comprehensive_val_sft_data.jsonl", 'r') as f:
                val_sft_pool.extend([json.loads(line) for line in f])



        # Shuffle to mix categories (Food/Clothing/Electronics)
        random.shuffle(train_sft_pool)
        random.shuffle(val_sft_pool)

        # Save to Master Directory
        out_dir = os.path.join(base_path, f"MASTER_{exp_name.upper()}")
        os.makedirs(out_dir, exist_ok=True)

        with open(os.path.join(out_dir, "train_sft.jsonl"), 'w') as f:
            for item in train_sft_pool: f.write(json.dumps(item) + '\n')
        with open(os.path.join(out_dir, "val_sft.jsonl"), 'w') as f:
            for item in val_sft_pool: f.write(json.dumps(item) + '\n')


        print(f"✅ {exp_name} Train: {len(train_sft_pool)} SFT | {len(val_sft_pool)} SFT")

create_SFT_experimental_files("/content/drive/My Drive/baselines/SFT/")


✅ BundleRec Train: 1998 SFT | 248 SFT
✅ LLM4Bear Train: 1886 SFT | 197 SFT


# The difference in number of samples can be explained by:

## In scenarios where LLM teacher models generally get things correct (i.e., no jaccard < 0.3), I show this example during SFT around half the time.

## This allows samples where product bundling is harder (i.e., disagreement with teacher models) is more highly represented.

# Thus, your SFT files will not be the same as mine, though the number of validation samples will tell you that they were created the same way.

In [ ]:

        # if len(rejected_indices) == 0 and train_jaccards[i][0] > 0.8:
        #     if random.random() < 0.5:
        #         SFT_samples.append({
        #                 "prompt": get_zero_shot(train_set[train_keys[i]]),
        #                 "answer": train_guys[i][0],
        #                 "score": train_jaccards[i][0] # Useful for analysis
        #             })


In [ ]:
from openai import OpenAI
import os

from google.colab import userdata
my_secret_key = userdata.get('API_KEY')


if my_secret_key:
    print("Token retrieved successfully.")
else:
    print("Token not found in Colab Secrets.")


client = OpenAI(
    # This is the default and can be omitted
    api_key = my_secret_key,
)

In [ ]:
def run_SFT(client, exp_subpath, master_path, num_epochs, model):
    print(f"\n🚀 Starting Full Pipeline for: {exp_subpath}")

    # --- 1. FILE UPLOAD ---
    train_sft_file = client.files.create(file=open(f"{master_path}{exp_subpath}train_sft.jsonl", "rb"), purpose="fine-tune")
    val_sft_file = client.files.create(file=open(f"{master_path}{exp_subpath}val_sft.jsonl", "rb"), purpose="fine-tune")

    # --- 2. STAGE 1: SFT (3 Epochs) ---
    sft_job = client.fine_tuning.jobs.create(
        training_file=train_sft_file.id,
        validation_file=val_sft_file.id,
        # model="gpt-4.1-mini-2025-04-14",
        # model = "gpt-4o-mini-2024-07-18",
        model=model,
        integrations=[{
            "type": "wandb",
            "wandb": {
                "project": "BundleRec-vs-LLM4Bear",
                "name": f"{exp_subpath.strip('/')}-sft"
            }
        }],
        method={
            "type": "supervised",
            "supervised": {
                "hyperparameters": {
                    "n_epochs": num_epochs,
                    "batch_size": 4,             # Increased from 1
                    "learning_rate_multiplier": 0.2 # Lowered slightly for stability
                }
            }
        },
        suffix=f"{exp_subpath.lower().replace('/', '').replace('_', '-')}-sft"
    )

# Directly do the fine-tuning task, with the provided SFT files.

In [ ]:
# master_path = "/content/drive/My Drive/baselines/SFT/"


master_path = "/content/LLM4BEAR/4_Bundle Generation/data/SFT/"
sub_paths = ["MASTER_LLM4BEAR/", "MASTER_BUNDLEREC/"]


for model in ["gpt-4o-mini-2024-07-18", "gpt-4.1-mini-2025-04-14"]:



    for path in sub_paths:
        job_id = run_SFT(client, path, master_path, num_epochs=3, model=model)
        print(f"Monitoring Job: https://platform.openai.com/finetune/{job_id}")



In [ ]:

async def finetuned_request(user, finetuned_model, system=None, seed_value=None):

    if system:
        message = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    else:
        message = [{"role": "user", "content": user}]

    # Reimplemented the robust retry loop with exponential backoff
    for delay_secs in (2**x for x in range(0, 3)):
        try:
            response = await async_client.chat.completions.create(
                model=finetuned_model,
                messages=message,
                temperature=0,
                max_tokens=1200,
                seed=seed_value
            )
            return response.choices[0].message.content.strip()
        except openai.OpenAIError as e:
            randomness_collision_avoidance = random.randint(0, 1000) / 1000.0
            sleep_dur = delay_secs + randomness_collision_avoidance
            print(f"Error: {e}. Retrying in {round(sleep_dur, 2)} seconds.")
            await asyncio.sleep(sleep_dur)

    # Return None if all retries fail
    return None


async def openai_finetuned(prompts, finetuned_model, system=None, batch_size=128, delay=5):

    results = []

    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        tasks = [
            finetuned_request(d["prompts"], finetuned_model, system=system, seed_value=42)
            for j, d in enumerate(batch)
        ]

        batch_results = await asyncio.gather(*tasks)
        results.extend(batch_results)
        print(f"✅ Sending batch {i // batch_size + 1} — sleeping for {delay}s...\n")
        await asyncio.sleep(delay)

    return results

In [ ]:
async def run_zero_shot_test(path, finetune_model_id, finetune_name):

    domain = path.split("/")[6]
    dataset = path.split("/")[5]

    item_titles_path = "item_titles.npy"
    session_bundles_deduplication_path = "session_bundles_deduplication.npy"
    session_items = "session_items.npy"
    topK_related_sessions_path = "TopK_related_sessions.npy"
    training_set_path = "training_set.npy"
    test_set_path = "test_set.npy"

    item_titles = np.load(path + item_titles_path, allow_pickle=True).tolist()
    session_bundles_deduplication = np.load(path + session_bundles_deduplication_path, allow_pickle=True).tolist()
    session_items = np.load(path + session_items, allow_pickle=True).tolist()
    topK_related_sessions = np.load(path + topK_related_sessions_path, allow_pickle=True).tolist()
    training_set = np.load(path + training_set_path, allow_pickle=True).tolist()
    test_set = np.load(path + test_set_path, allow_pickle=True).tolist()

    test_keys = list(test_set.keys())

    zero_shot_prompt_list = [{"prompts": get_zero_shot(test_set[i])} for i in test_keys]

    system_message = "You are an **Expert E-commerce Analyst and Product Bundler**. Your task is to receive a sequence of products and accurately organise them into logical, desirable bundles."

    zero_shot_responses = await openai_finetuned(zero_shot_prompt_list, finetune_model_id, system=system_message)

    print()
    print(zero_shot_responses[0])
    print()


    with open(f"/content/drive/MyDrive/baselines/SFT/{dataset}/{domain}/" + f'finetuned_responses_{finetune_name}.pkl', 'wb') as f:
        pkl.dump(zero_shot_responses, f)




In [ ]:

from google.colab import userdata
# my_secret_key = userdata.get('API_KEY')


if my_secret_key:
  print("Token retrieved successfully.")
else:
  print("Token not found in Colab Secrets.")


client = OpenAI(
    # This is the default and can be omitted
    api_key = my_secret_key,
)


async_client = AsyncOpenAI(api_key = my_secret_key,
)  # make sure this is your actual key

# You have to use your own finetuned model.

In [ ]:
electronic_bundlerec_path = '/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/electronic/'
clothing_bundlerec_path = '/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/clothing/'
food_bundlerec_path = '/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/food/'

electronic_llm4bear_path = '/content/LLM4BEAR/4_Bundle Generation/data/llm4bear/electronic/'
clothing_llm4bear_path = '/content/LLM4BEAR/4_Bundle Generation/data/llm4bear/clothing/'
food_llm4bear_path = '/content/LLM4BEAR/4_Bundle Generation/data/llm4bear/food/'


bundlerec_finetuned = ["ft:gpt-4o-mini-2024-07-18:project:master-bundlerec-sft:{INSERT FINETUNED MODEL}", "ft:gpt-4.1-mini-2025-04-14:project:master-bundlerec-sft:{INSERT FINETUNED MODEL}"]
llm4bear_finetuned = ["ft:gpt-4o-mini-2024-07-18:project:master-llm4bear-sft:{INSERT FINETUNED MODEL}", "ft:gpt-4.1-mini-2025-04-14:project:master-llm4bear-sft:{INSERT FINETUNED MODEL}"]

finetune_models = ["4o-mini", "4.1-mini"]

for i in range(2):

    await run_zero_shot_test(path=electronic_bundlerec_path, finetune_model_id=bundlerec_finetuned[i], dataset_name=f"{finetune_models[i]}_brec_tuned")

    await run_zero_shot_test(path=clothing_bundlerec_path, finetune_model_id=bundlerec_finetuned[i], dataset_name=f"{finetune_models[i]}_brec_tuned")

    await run_zero_shot_test(path=food_bundlerec_path, finetune_model_id=bundlerec_finetuned[i], dataset_name=f"{finetune_models[i]}_brec_tuned")

    await run_zero_shot_test(path=electronic_llm4bear_path, finetune_model_id=llm4bear_finetuned[i], dataset_name=f"{finetune_models[i]}_bear_tuned")

    await run_zero_shot_test(path=clothing_llm4bear_path, finetune_model_id=llm4bear_finetuned[i], dataset_name=f"{finetune_models[i]}_bear_tuned")

    await run_zero_shot_test(path=food_llm4bear_path, finetune_model_id=llm4bear_finetuned[i], dataset_name=f"{finetune_models[i]}_bear_tuned")